## 1. Import Libraries and Setup

In [4]:
import os
import lancedb
import subprocess
import requests
import time
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv

# LlamaIndex Core Components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core.ingestion import IngestionPipeline

# Embeddings and vector store
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# LLM integrations
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.ollama import Ollama

# Async support for notebooks
import nest_asyncio
nest_asyncio.apply()
load_dotenv()

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Data Preparation and Loading

In [5]:
def prepare_data(num_samples = 100):
    """
    Load dataset and create document files
    """
    print(f"Loading {num_samples} personas from dataset...")
    
    # Load the personas dataset
    dataset = load_dataset("dvilasuero/finepersonas-v0.1-tiny", split="train")
    
    # Create data directory
    Path("data").mkdir(parents=True, exist_ok=True)
    
    # Save personas as text files and create Document objects
    documents = []
    for i, persona in enumerate(dataset.select(range(min(num_samples, len(dataset))))):
        # Create Document objects for LlamaIndex
        doc = Document(
            text = persona["persona"],  # type: ignore[index]
            metadata = {
                "persona_id": i,
                "source": "finepersonas-dataset"
            } 
        )
        documents.append(doc)
        
        # Optionally save to files as well
        with open(Path("data") / f"persona_{i}.txt", "w", encoding="utf-8") as f:
            f.write(persona["persona"])  # type: ignore[index]
            
    print(f"Prepared {len(documents)} documents")
    return documents

documents = prepare_data(num_samples=100)

Loading 100 personas from dataset...
Prepared 100 documents


## 3. LanceDB Vector Store Setup

In [6]:
def setup_lancedb_store(table_name="personas_rag"):
    """
    Initialize LanceDB and create/connect to a table
    """
    print("Setting up LanceDB connection...")
    
    # Create/connect to LanceDB
    db = lancedb.connect("./lancedb_data")
    
    # LlamaIndex will handle table creation with proper schema
    print(f"Connected to LanceDB, table: {table_name}")
    
    return db, table_name

db, table_name = setup_lancedb_store()

Setting up LanceDB connection...
Connected to LanceDB, table: personas_rag


## 5. Vector Embeddings and Ingestion Pipeline

In [9]:
async def create_and_populate_index(documents, db, table_name):
    """
    Create ingestion pipeline and populate LanceDB with embeddings. 
    """
    print("Creating Embedding Model and Ingestion Pipeline...")
    
    # Initialize embedding model
    embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5"
    )
    
    # Create LanceDB vector store
    vector_store = LanceDBVectorStore(
        uri="./lancedb_data",
        table_name=table_name,
        mode="overwrite"    # overwrite existing table
    )
    
    # Create ingestion pipeline
    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_size=100, chunk_overlap=20),
            embed_model
        ],
        vector_store=vector_store
    )
    
    print("Processing documents and creating embeddings...")
    # Run the pipeline to process documents and store in LanceDB
    nodes = await pipeline.arun(documents = documents)
    print(f"Successfully processed {len(nodes)} text chunks")
    
    return vector_store, embed_model

vector_store, embed_model = await create_and_populate_index(documents, db, table_name)

Creating Embedding Model and Ingestion Pipeline...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8683.86it/s]


Processing documents and creating embeddings...
Successfully processed 100 text chunks
